In [ ]:
import os 
import pandas as pd

# === 1. 파일 경로 설정 ===
base_path = "C:/Jimin/cg_suri_z-code360/code_analysis/medi_code/"
file_suga = os.path.join(base_path,"★수가반영내역(25.8.1.기준)_전체판.xlsx")
file_kcd = r"C:\Jimin\cg_suri_z-code360\code_analysis\medi_code\KCD-9 DB masterfile_250701_20250701010653.xlsx"
file_diag = r"C:\Jimin\cg_suri_z-code360\code_analysis\medi_code\배포용 상병마스터_240101(2).xlsx"

# === 2. 각 시트 로딩 ===
df_suga = pd.read_excel(file_suga, sheet_name="의치과_급여_전체")
# df_kcd = pd.read_excel(file_kcd, sheet_name="KCD-8 DB Masterfile")
df_kcd = pd.read_excel("KCD-9 DB masterfile_250701.xlsx", sheet_name="KCD-8 DB Masterfile")

df_diag = pd.read_excel(file_diag, sheet_name="상병분류기호(완전코드)")

# === 3. 컬럼 정리 ===
df_kcd = df_kcd.rename(columns={
    '상병코드': 'KCD코드',
    '한글진단명': '질환명(한글)',
    '영문진단명': '질환명(영문)'
})[['KCD코드', '질환명(한글)', '질환명(영문)']]

df_diag = df_diag.rename(columns={
    df_diag.columns[0]: '주상병코드'
})[['주상병코드']].dropna()

# === 4. 상병코드 ↔ 질환명 병합 ===
df_kcd_diag = pd.merge(df_diag, df_kcd, left_on='주상병코드', right_on='KCD코드', how='left')

# === 5. 관심 질환 필터링 ===
keywords = ['당뇨', '암', '심근경색', '폐렴']  # ← 형님이 원하는 질환명 추가 가능
df_filtered = df_kcd_diag[df_kcd_diag['질환명(한글)'].str.contains('|'.join(keywords), na=False)]

# === 6. 수가코드 정리 ===
df_suga = df_suga.rename(columns={
    df_suga.columns[0]: '수가코드',
    df_suga.columns[1]: '수가명'
})[['수가코드', '수가명']].dropna()

# === 7. 수가코드를 랜덤 매핑 (임시) ===
df_filtered = df_filtered.reset_index(drop=True)
df_filtered['수가코드'] = df_suga['수가코드'].sample(n=len(df_filtered), replace=True).values
df_filtered['수가명'] = df_suga['수가명'].sample(n=len(df_filtered), replace=True).values

# === 8. 저장 ===
df_filtered.to_csv("질환_상병_수가코드_매핑.csv", index=False, encoding='utf-8-sig')
print("✅ 저장 완료: '질환_상병_수가코드_매핑.csv'")


KeyError: "None of [Index(['KCD코드', '질환명(한글)', '질환명(영문)'], dtype='object')] are in the [columns]"

In [3]:
print(df_kcd.columns.tolist())


['Unnamed: 0', 'KCD-9 DB Masterfile', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']
